In [1]:
from six.moves import urllib
opener = urllib.request.build_opener()
opener.addheaders = [('User-agent', 'Mozilla/5.0')]
urllib.request.install_opener(opener)

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets, transforms

# Define a transform to normalize the data
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,)),
                              ])
# Download and load the training data
trainset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

In [3]:
# Build a feed-forward network
model = nn.Sequential(nn.Linear(784, 128),
                      nn.ReLU(),
                      nn.Linear(128, 64),
                      nn.ReLU(),
                      nn.Linear(64, 10))

# Define the loss
criterion = nn.CrossEntropyLoss()

# Get our data
images, labels = next(iter(trainloader))
# Flatten images
images = images.view(images.shape[0], -1)

# Forward pass, get our logits
logits = model(images)
# Calculate the loss with the logits and the labels
loss = criterion(logits, labels)

print(loss)

tensor(2.2996, grad_fn=<NllLossBackward0>)


In [4]:
x = torch.randn(2,2, requires_grad=True)
print(x)

tensor([[ 0.9231,  1.6934],
        [ 0.7945, -0.5971]], requires_grad=True)


In [5]:
y = x**2
print(y)

tensor([[0.8521, 2.8675],
        [0.6312, 0.3565]], grad_fn=<PowBackward0>)


In [6]:
print(y.grad_fn)


In [7]:
z = y.mean()
print(z)

tensor(1.1768, grad_fn=<MeanBackward0>)


In [8]:
print(x.grad)

None


In [9]:
z.backward()
print(x.grad)
print(x/2)

tensor([[ 0.4615,  0.8467],
        [ 0.3972, -0.2985]])
tensor([[ 0.4615,  0.8467],
        [ 0.3972, -0.2985]], grad_fn=<DivBackward0>)


In [10]:
# Build a feed-forward network
model = nn.Sequential(nn.Linear(784, 128),
                      nn.ReLU(),
                      nn.Linear(128, 64),
                      nn.ReLU(),
                      nn.Linear(64, 10),
                      nn.LogSoftmax(dim=1))

criterion = nn.NLLLoss()
images, labels = next(iter(trainloader))
images = images.view(images.shape[0], -1)

logps = model(images)
loss = criterion(logps, labels)

In [11]:
print('Before backward pass: \n', model[0].weight.grad)

loss.backward()

print('After backward pass: \n', model[0].weight.grad)

Before backward pass: 
 None
After backward pass: 
 tensor([[-0.0009, -0.0009, -0.0009,  ..., -0.0009, -0.0009, -0.0009],
        [ 0.0005,  0.0005,  0.0005,  ...,  0.0005,  0.0005,  0.0005],
        [ 0.0002,  0.0002,  0.0002,  ...,  0.0002,  0.0002,  0.0002],
        ...,
        [ 0.0029,  0.0029,  0.0029,  ...,  0.0029,  0.0029,  0.0029],
        [ 0.0004,  0.0004,  0.0004,  ...,  0.0004,  0.0004,  0.0004],
        [-0.0015, -0.0015, -0.0015,  ..., -0.0015, -0.0015, -0.0015]])


In [12]:
from torch import optim

# Optimizers require the parameters to optimize and a learning rate
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [13]:
print('Initial weights - ', model[0].weight)

images, labels = next(iter(trainloader))
images.resize_(64, 784)

# Clear the gradients, do this because gradients are accumulated
optimizer.zero_grad()

# Forward pass, then backward pass, then update weights
output = model(images)
loss = criterion(output, labels)
loss.backward()
print('Gradient -', model[0].weight.grad)

Initial weights -  Parameter containing:
tensor([[ 0.0186,  0.0333, -0.0351,  ...,  0.0043,  0.0355,  0.0067],
        [-0.0066,  0.0219,  0.0207,  ...,  0.0059, -0.0208,  0.0109],
        [ 0.0120,  0.0325, -0.0072,  ..., -0.0041,  0.0285,  0.0281],
        ...,
        [-0.0170,  0.0018, -0.0307,  ..., -0.0007, -0.0184, -0.0227],
        [ 0.0057, -0.0328, -0.0120,  ...,  0.0307,  0.0207,  0.0125],
        [ 0.0333,  0.0274,  0.0087,  ...,  0.0109, -0.0290, -0.0255]],
       requires_grad=True)
Gradient - tensor([[ 0.0006,  0.0006,  0.0006,  ...,  0.0006,  0.0006,  0.0006],
        [ 0.0076,  0.0076,  0.0076,  ...,  0.0076,  0.0076,  0.0076],
        [ 0.0016,  0.0016,  0.0016,  ...,  0.0016,  0.0016,  0.0016],
        ...,
        [ 0.0005,  0.0005,  0.0005,  ...,  0.0005,  0.0005,  0.0005],
        [ 0.0010,  0.0010,  0.0010,  ...,  0.0010,  0.0010,  0.0010],
        [-0.0001, -0.0001, -0.0001,  ..., -0.0001, -0.0001, -0.0001]])


In [14]:
# Take an update step and few the new weights
optimizer.step()
print('Updated weights - ', model[0].weight)

Updated weights -  Parameter containing:
tensor([[ 0.0186,  0.0333, -0.0351,  ...,  0.0043,  0.0355,  0.0067],
        [-0.0067,  0.0218,  0.0206,  ...,  0.0059, -0.0209,  0.0108],
        [ 0.0120,  0.0325, -0.0073,  ..., -0.0042,  0.0285,  0.0281],
        ...,
        [-0.0170,  0.0018, -0.0307,  ..., -0.0007, -0.0184, -0.0227],
        [ 0.0057, -0.0328, -0.0120,  ...,  0.0307,  0.0207,  0.0125],
        [ 0.0333,  0.0274,  0.0087,  ...,  0.0109, -0.0290, -0.0255]],
       requires_grad=True)


In [15]:
model = nn.Sequential(nn.Linear(784, 128),
                      nn.ReLU(),
                      nn.Linear(128, 64),
                      nn.ReLU(),
                      nn.Linear(64, 10),
                      nn.LogSoftmax(dim=1))

criterion = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.003)

epochs = 5
for e in range(epochs):
    running_loss = 0
    for images, labels in trainloader:
        # Flatten MNIST images into a 784 long vector
        images = images.view(images.shape[0], -1)
    
        # TODO: Training pass
        optimizer.zero_grad()
        
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    else:
        print(f"Training loss: {running_loss/len(trainloader)}")

Training loss: 1.921538968457342
Training loss: 0.8755537231148941
Training loss: 0.5292594703529944
Training loss: 0.4290602695522532
Training loss: 0.38350119699102475


In [ ]:
%matplotlib inline
import helper

images, labels = next(iter(trainloader))

img = images[0].view(1, 784)
# Turn off gradients to speed up this part
with torch.no_grad():
    logps = model(img)

# Output of the network are log-probabilities, need to take exponential for probabilities
ps = torch.exp(logps)
helper.view_classify(img.view(1, 28, 28), ps)